In [1]:
# Install and import necessary NLP libraries
!pip install nltk pandas scikit-learn -q
import nltk
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')

print("Libraries and dependencies loaded successfully!")

Libraries and dependencies loaded successfully!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [2]:
# Create a realistic student feedback dataset across multiple departments
data = {
    'Feedback_ID': ['S101', 'S102', 'S103', 'S104', 'S105', 'S106', 'S107', 'S108'],
    'Department': ['Computer Science', 'Mathematics', 'Physics', 'Computer Science', 'Mathematics', 'Physics', 'Computer Science', 'Mathematics'],
    'Rating': [4, 2, 5, 1, 4, 3, 2, 5],
    'Feedback_Text': [
        "Professor Adams explains complex algorithms very clearly, but the lab assignments are way too rushed.",
        "The textbook is completely outdated and the online LMS portal crashes constantly during quizzes.",
        "Hands-on experiments in the physics lab session really helped me understand quantum mechanics!",
        "Lectures are way too fast and monotone, making it difficult to stay awake or take proper notes.",
        "Great practical programming projects and very approachable instructors during office hours.",
        "The projector in the lecture hall broke down multiple times, delaying the presentation.",
        "Grading criteria for the programming assignment were vague and unclear.",
        "The syllabus is well-structured and the study materials provided are exceptional."
    ]
}

df = pd.DataFrame(data)
df.head()

,Feedback_ID,Department,Rating,Feedback_Text
0,S101,Computer Science,4,Professor Adams explains complex algorithms ve...
1,S102,Mathematics,2,The textbook is completely outdated and the on...
2,S103,Physics,5,Hands-on experiments in the physics lab sessio...
3,S104,Computer Science,1,"Lectures are way too fast and monotone, making..."
4,S105,Mathematics,4,Great practical programming projects and very ...


In [3]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lowercase text
    text = text.lower()
    # 2. Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # 3. Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # 4. Tokenization, Stopword Removal, & Lemmatization
    tokens = text.split()
    cleaned_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]

    return " ".join(cleaned_tokens)

# Apply preprocessing
df['Cleaned_Text'] = df['Feedback_Text'].apply(preprocess_text)
df[['Feedback_ID', 'Feedback_Text', 'Cleaned_Text']]

,Feedback_ID,Feedback_Text,Cleaned_Text
0,S101,Professor Adams explains complex algorithms ve...,professor adam explains complex algorithm clea...
1,S102,The textbook is completely outdated and the on...,textbook completely outdated online lm portal ...
2,S103,Hands-on experiments in the physics lab sessio...,handson experiment physic lab session really h...
3,S104,"Lectures are way too fast and monotone, making...",lecture way fast monotone making difficult sta...
4,S105,Great practical programming projects and very ...,great practical programming project approachab...
5,S106,The projector in the lecture hall broke down m...,projector lecture hall broke multiple time del...
6,S107,Grading criteria for the programming assignmen...,grading criterion programming assignment vague...
7,S108,The syllabus is well-structured and the study ...,syllabus wellstructured study material provide...


In [4]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

# Apply sentiment classification
df['Sentiment'] = df['Feedback_Text'].apply(get_sentiment)
df['Compound_Score'] = df['Feedback_Text'].apply(lambda x: sia.polarity_scores(x)['compound'])

df[['Feedback_ID', 'Rating', 'Sentiment', 'Compound_Score', 'Feedback_Text']]

,Feedback_ID,Rating,Sentiment,Compound_Score,Feedback_Text
0,S101,4,Positive,0.2492,Professor Adams explains complex algorithms ve...
1,S102,2,Neutral,0.0000,The textbook is completely outdated and the on...
2,S103,5,Neutral,0.0000,Hands-on experiments in the physics lab sessio...
3,S104,1,Negative,-0.3612,"Lectures are way too fast and monotone, making..."
4,S105,4,Positive,0.6249,Great practical programming projects and very ...
5,S106,3,Negative,-0.4215,The projector in the lecture hall broke down m...
6,S107,2,Negative,-0.3400,Grading criteria for the programming assignmen...
7,S108,5,Neutral,0.0000,The syllabus is well-structured and the study ...


In [5]:
# Vectorize text using TF-IDF for Topic Modeling
tfidf_vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['Cleaned_Text'])

# Fit Latent Dirichlet Allocation (LDA) to discover latent student concerns
lda_model = LatentDirichletAllocation(n_components=3, random_state=42)
lda_model.fit(tfidf_matrix)

# Display top keywords per topic (Concern clusters)
feature_names = tfidf_vectorizer.get_feature_names_out()
print("=== Discovered Student Concern Topics (LDA Model) ===")
for topic_idx, topic in enumerate(lda_model.components_):
    top_keywords = [feature_names[i] for i in topic.argsort()[:-4:-1]]
    print(f"Topic {topic_idx + 1}: {', '.join(top_keywords)}")

=== Discovered Student Concern Topics (LDA Model) ===
Topic 1: unclear, vague, criterion
Topic 2: practical, project, office
Topic 3: way, lab, quiz


In [6]:
# Define academic aspect keywords
concern_keywords = {
    'Teaching Methods': ['lecture', 'explain', 'clear', 'fast', 'pace', 'instructor', 'professor', 'monotone'],
    'Curriculum / Syllabus': ['textbook', 'syllabus', 'content', 'outdated', 'difficult', 'theory', 'subject', 'material'],
    'Lab / Assignments': ['assignment', 'lab', 'project', 'practical', 'experiment', 'workshop', 'rushed', 'grading'],
    'Infrastructure / Portal': ['portal', 'lms', 'computer', 'projector', 'wifi', 'equipment', 'online', 'app', 'hall']
}

def extract_concerns(text):
    matched_concerns = []
    text_lower = text.lower()
    for category, keywords in concern_keywords.items():
        if any(kw in text_lower for kw in keywords):
            matched_concerns.append(category)
    return matched_concerns if matched_concerns else ['General Experience']

# Map concerns
df['Concerns'] = df['Feedback_Text'].apply(extract_concerns)
df[['Feedback_ID', 'Concerns', 'Feedback_Text']]

,Feedback_ID,Concerns,Feedback_Text
0,S101,"[Teaching Methods, Lab / Assignments]",Professor Adams explains complex algorithms ve...
1,S102,"[Curriculum / Syllabus, Infrastructure / Portal]",The textbook is completely outdated and the on...
2,S103,[Lab / Assignments],Hands-on experiments in the physics lab sessio...
3,S104,"[Teaching Methods, Curriculum / Syllabus]","Lectures are way too fast and monotone, making..."
4,S105,"[Teaching Methods, Lab / Assignments, Infrastr...",Great practical programming projects and very ...
5,S106,"[Teaching Methods, Lab / Assignments, Infrastr...",The projector in the lecture hall broke down m...
6,S107,"[Teaching Methods, Lab / Assignments]",Grading criteria for the programming assignmen...
7,S108,"[Curriculum / Syllabus, Lab / Assignments]",The syllabus is well-structured and the study ...


In [7]:
# Summary statistics and analytics
print("=== Sentiment Breakdown ===")
print(df['Sentiment'].value_counts(), "\n")

print("=== Department-wise Average Ratings ===")
print(df.groupby('Department')['Rating'].mean(), "\n")

print("=== Actionable Administrative Recommendations ===")
print("1. Lab & Assignment Pacing: Review computer science coding assignment deadlines and clarify grading rubrics.")
print("2. Infrastructure Upgrade: Inspect lecture hall projectors and upgrade the LMS portal stability during exams.")
print("3. Faculty Development: Provide workshops for instructors on lecture pacing and student engagement techniques.")

# View final consolidated analysis dataframe
df[['Feedback_ID', 'Department', 'Rating', 'Sentiment', 'Concerns', 'Feedback_Text']]

=== Sentiment Breakdown ===
Sentiment
Neutral     3
Negative    3
Positive    2
Name: count, dtype: int64 

=== Department-wise Average Ratings ===
Department
Computer Science    2.333333
Mathematics         3.666667
Physics             4.000000
Name: Rating, dtype: float64 

=== Actionable Administrative Recommendations ===
1. Lab & Assignment Pacing: Review computer science coding assignment deadlines and clarify grading rubrics.
2. Infrastructure Upgrade: Inspect lecture hall projectors and upgrade the LMS portal stability during exams.
3. Faculty Development: Provide workshops for instructors on lecture pacing and student engagement techniques.


,Feedback_ID,Department,Rating,Sentiment,Concerns,Feedback_Text
0,S101,Computer Science,4,Positive,"[Teaching Methods, Lab / Assignments]",Professor Adams explains complex algorithms ve...
1,S102,Mathematics,2,Neutral,"[Curriculum / Syllabus, Infrastructure / Portal]",The textbook is completely outdated and the on...
2,S103,Physics,5,Neutral,[Lab / Assignments],Hands-on experiments in the physics lab sessio...
3,S104,Computer Science,1,Negative,"[Teaching Methods, Curriculum / Syllabus]","Lectures are way too fast and monotone, making..."
4,S105,Mathematics,4,Positive,"[Teaching Methods, Lab / Assignments, Infrastr...",Great practical programming projects and very ...
5,S106,Physics,3,Negative,"[Teaching Methods, Lab / Assignments, Infrastr...",The projector in the lecture hall broke down m...
6,S107,Computer Science,2,Negative,"[Teaching Methods, Lab / Assignments]",Grading criteria for the programming assignmen...
7,S108,Mathematics,5,Neutral,"[Curriculum / Syllabus, Lab / Assignments]",The syllabus is well-structured and the study ...
